In [ ]:
import pandas as pd

df = pd.read_csv("garak_apple_prices.csv")
df["date"] = pd.to_datetime(df["date"])

# 2019~2023 제거
df = df[~df["date"].dt.year.isin([2019, 2020, 2021, 2022, 2023])]

# 10 키로상자만 사용 (단위 통일)
df = df[df["unit"] == "10 키로상자"]

print("연도별 건수:")
print(df.groupby(df["date"].dt.year)["date"].count())

# 등급별 저장
for grade in ["특", "상", "중", "하"]:
    g = df[df["grade"] == grade].sort_values("date").reset_index(drop=True)
    
    # gap 30일 이상이면 window 분리
    g["gap"] = g["date"].diff().dt.days.fillna(0)
    g["window_id"] = (g["gap"] > 30).cumsum()
    
    # window별 건수 출력
    print(f"\n[{grade}] window 분포:")
    print(g.groupby("window_id")["date"].agg(["count", "min", "max"]))
    
    # 각 window를 별도 저장 (학습 시 window별로 초기화)
    for wid, wdf in g.groupby("window_id"):
        wdf = wdf.drop(columns=["gap", "window_id"]).reset_index(drop=True)
        fname = f"garak_apple_{grade}_w{wid}.csv"
        wdf.to_csv(fname, index=False, encoding="utf-8-sig")
        print(f"  저장: {fname} ({len(wdf)}건)")

In [ ]:
for grade in ["특", "상", "중", "하"]:
    dfs = []
    for w in [0, 1, 2]:
        tmp = pd.read_csv(f"garak_apple_{grade}_w{w}.csv")
        dfs.append(tmp)
    
    df = pd.concat(dfs, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    df = df[["date", "grade", "price_per_kg"]]
    
    df.to_csv(f"garak_apple_{grade}_final.csv", index=False, encoding="utf-8-sig")
    print(f"[{grade}] {len(df)}건")

In [ ]:
df_w2 = pd.read_csv("garak_apple_상_w2.csv")
df_w2["date"] = pd.to_datetime(df_w2["date"])

# w2 앞 70% → train에 합치기
# w2 뒤 30% → validation
split = int(len(df_w2) * 0.7)
w2_train = df_w2.iloc[:split]
w2_valid = df_w2.iloc[split:]

print(f"w2 train: {w2_train['date'].min()} ~ {w2_train['date'].max()}")
print(f"w2 valid: {w2_valid['date'].min()} ~ {w2_valid['date'].max()}")

FileNotFoundError: [Errno 2] No such file or directory: 'garak_apple_상_w2.csv'

In [5]:
import pandas as pd
import pyarrow as pa
from pathlib import Path

ARROW_DIR = Path("arrow_data")

for grade in ["상", "중"]:
    for w in [0, 1, 2]:
        df = pd.read_csv(f"preprocessed_data/garak_apple_{grade}_w{w}.csv")
        df = df.sort_values("date").reset_index(drop=True)
        
        split = int(len(df) * 0.9)
        train_df = df.iloc[:split]
        valid_df = df.iloc[split:]
        
        print(f"[{grade}] w{w} → train: {len(train_df)}건 / valid: {len(valid_df)}건")
        
        # train arrow
        table = pa.table({
            "start": pa.array([pd.Timestamp("2000-01-01")], type=pa.timestamp("s")),
            "target": pa.array([train_df["price_per_kg"].astype(float).values.tolist()], type=pa.list_(pa.float64()))
        })
        with pa.ipc.new_file(ARROW_DIR / f"{grade}_w{w}_train.arrow", table.schema) as writer:
            writer.write_table(table)
        
        # valid arrow
        table = pa.table({
            "start": pa.array([pd.Timestamp("2000-01-01")], type=pa.timestamp("s")),
            "target": pa.array([valid_df["price_per_kg"].astype(float).values.tolist()], type=pa.list_(pa.float64()))
        })
        with pa.ipc.new_file(ARROW_DIR / f"{grade}_w{w}_valid.arrow", table.schema) as writer:
            writer.write_table(table)

print("완료")

[상] w0 → train: 284건 / valid: 32건
[상] w1 → train: 179건 / valid: 20건
[상] w2 → train: 229건 / valid: 26건
[중] w0 → train: 284건 / valid: 32건
[중] w1 → train: 179건 / valid: 20건
[중] w2 → train: 229건 / valid: 26건
완료


In [1]:
import pandas as pd
df = pd.read_csv("preprocessed_data/test.csv")
print(df.columns.tolist())
print(df.head())
print(df.shape)

['date', 'market', 'item', 'grade', 'unit', 'price', 'kg_unit', 'price_per_kg', 'source']
         date market    item grade     unit  price  kg_unit  price_per_kg  \
0  2026-01-03   가락시장  사과 미시마     상  10 키로상자  60961     10.0          6096   
1  2026-01-03   가락시장  사과 미시마     중  10 키로상자  38974     10.0          3897   
2  2026-01-03   가락시장  사과 미시마     특  10 키로상자  72953     10.0          7295   
3  2026-01-03   가락시장  사과 미시마     하  10 키로상자  22608     10.0          2261   
4  2026-01-05   가락시장  사과 미시마     상  10 키로상자  71107     10.0          7111   

                                      source  
0  https://www.garakprice.com/pum_detail.php  
1  https://www.garakprice.com/pum_detail.php  
2  https://www.garakprice.com/pum_detail.php  
3  https://www.garakprice.com/pum_detail.php  
4  https://www.garakprice.com/pum_detail.php  
(552, 9)


In [ ]:
import pandas as pd
import torch
from chronos import ChronosPipeline
import numpy as np

# 테스트 데이터 로드
test_df = pd.read_csv("preprocessed_data/test.csv")
test_df["date"] = pd.to_datetime(test_df["date"])
test_df = test_df.sort_values("date").reset_index(drop=True)

# train 데이터 (context로 사용)
train_df = pd.read_csv("preprocessed_data/train_상.csv")
train_df["date"] = pd.to_datetime(train_df["date"])
train_df = train_df.sort_values("date").reset_index(drop=True)

# 모델 로드 (baseline fine-tuned)
pipeline = ChronosPipeline.from_pretrained(
    "finetuned_model/chronos_baseline/run-0/checkpoint-final",
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    dtype=torch.float32,
)

# 상 등급만 테스트
grade = "상"
test_grade = test_df[test_df["grade"] == grade].sort_values("date").reset_index(drop=True)
actual = test_grade["price_per_kg"].values

# train 데이터를 context로 사용
context = torch.tensor(
    train_df[train_df["grade"] == grade]["price_per_kg"].values,
    dtype=torch.float32
)

# 예측
forecast = pipeline.predict(context, prediction_length=len(actual), num_samples=100)
median = forecast[0].median(dim=0).values.numpy()

# 평가
mae = np.mean(np.abs(actual - median))
mape = np.mean(np.abs((actual - median) / actual)) * 100

print(f"[{grade}등급] MAE: {mae:.2f}, MAPE: {mape:.2f}%")
print(f"실제 가격 범위: {actual.min():.0f} ~ {actual.max():.0f}")
print(f"예측 가격 범위: {median.min():.0f} ~ {median.max():.0f}")

In [ ]:
import pandas as pd
import torch
from chronos import ChronosPipeline
import numpy as np

# 데이터 로드
test_df = pd.read_csv("preprocessed_data/test.csv")
test_df["date"] = pd.to_datetime(test_df["date"])

train_df = pd.read_csv("preprocessed_data/train_상.csv")
train_df["date"] = pd.to_datetime(train_df["date"])

grade = "상"
context = torch.tensor(
    train_df[train_df["grade"] == grade]["price_per_kg"].values,
    dtype=torch.float32
)
test_grade = test_df[test_df["grade"] == grade].sort_values("date").reset_index(drop=True)
actual = test_grade["price_per_kg"].values
pred_len = len(actual)

def evaluate(pipeline, context, actual, name):
    forecast = pipeline.predict(context, prediction_length=pred_len, num_samples=100)
    median = forecast[0].median(dim=0).values.numpy()
    mae  = np.mean(np.abs(actual - median))
    mape = np.mean(np.abs((actual - median) / actual)) * 100
    print(f"[{name}] MAE: {mae:.2f} | MAPE: {mape:.2f}%")
    return median

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Zero-shot
print("Zero-shot 로드 중...")
zeroshot = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-mini",
    device_map=device,
    dtype=torch.float32,
)
pred_zero = evaluate(zeroshot, context, actual, "Zero-shot")

# 2. Baseline fine-tuned
print("Baseline fine-tuned 로드 중...")
baseline = ChronosPipeline.from_pretrained(
    "finetuned_model/chronos_baseline/run-0/checkpoint-final",
    device_map=device,
    dtype=torch.float32,
)
pred_base = evaluate(baseline, context, actual, "Baseline fine-tuned")

# 3. Extended fine-tuned
print("Extended fine-tuned 로드 중...")
extended = ChronosPipeline.from_pretrained(
    "finetuned_model/chronos_extended/run-0/checkpoint-final",
    device_map=device,
    dtype=torch.float32,
)
pred_ext = evaluate(extended, context, actual, "Extended fine-tuned")

In [8]:
from timesfm import TimesFM_2p5_200M_torch

print(dir(TimesFM_2p5_200M_torch))

['DEFAULT_REPO_ID', 'WEIGHTS_FILENAME', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_decode_arg', '_encode_arg', '_from_pretrained', '_hub_mixin_coders', '_hub_mixin_config', '_hub_mixin_info', '_hub_mixin_init_parameters', '_hub_mixin_inject_config', '_hub_mixin_jsonable_custom_types', '_hub_mixin_jsonable_default_values', '_is_jsonable', '_load_as_pickle', '_load_as_safetensor', '_save_pretrained', 'compile', 'compiled_decode', 'forecast', 'forecast_config', 'forecast_with_covariates', 'from_pretrained', 'generate_model_card', 'global_batch_size', 'load_checkpoint', 'push_to_hub', 'save_pretrained']


In [ ]:
import pandas as pd
import torch
import numpy as np

from chronos import ChronosPipeline
from uni2ts.model.moirai import MoiraiForecast


# =========================
# 데이터 로드
# =========================

test_df = pd.read_csv("preprocessed_data/test.csv")
test_df["date"] = pd.to_datetime(test_df["date"])

train_df = pd.read_csv("preprocessed_data/train_상.csv")
train_df["date"] = pd.to_datetime(train_df["date"])

grade = "상"

context = torch.tensor(
    train_df[train_df["grade"] == grade]["price_per_kg"].values,
    dtype=torch.float32
)

test_grade = (
    test_df[test_df["grade"] == grade]
    .sort_values("date")
    .reset_index(drop=True)
)

actual = test_grade["price_per_kg"].values
pred_len = len(actual)

device = "cuda" if torch.cuda.is_available() else "cpu"


# =========================
# Chronos 평가
# =========================

def evaluate_chronos(model, context, actual, name):

    forecast = model.predict(
        context,
        prediction_length=len(actual),
        num_samples=100,
    )

    pred = forecast[0].median(dim=0).values.cpu().numpy()

    mae = np.mean(np.abs(actual - pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100

    print(f"[{name}]")
    print(f"MAE  : {mae:.4f}")
    print(f"MAPE : {mape:.4f}%\n")

    return pred


# =========================
# Moirai 평가
# =========================

def evaluate_moirai(model, context, actual, name):

    with torch.no_grad():

        forecast = model.predict(
            past_target=context.unsqueeze(0).to(device),
            prediction_length=len(actual),
            num_samples=100,
        )

    pred = (
        forecast.median(dim=1)
        .values
        .squeeze()
        .cpu()
        .numpy()
    )

    mae = np.mean(np.abs(actual - pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100

    print(f"[{name}]")
    print(f"MAE  : {mae:.4f}")
    print(f"MAPE : {mape:.4f}%\n")

    return pred


# =========================
# Chronos Zero-shot
# =========================

print("Chronos Zero-shot 로드 중...")

zeroshot = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-mini",
    device_map=device,
    dtype=torch.float32,
)

pred_zero = evaluate_chronos(
    zeroshot,
    context,
    actual,
    "Chronos Zero-shot"
)


# =========================
# Chronos Baseline FT
# =========================

print("Chronos Baseline FT 로드 중...")

baseline = ChronosPipeline.from_pretrained(
    "finetuned_model/chronos_baseline/run-0/checkpoint-6000",
    device_map=device,
    dtype=torch.float32,
)

pred_base = evaluate_chronos(
    baseline,
    context,
    actual,
    "Chronos Baseline FT"
)


# =========================
# Chronos Extended FT
# =========================

print("Chronos Extended FT 로드 중...")

extended = ChronosPipeline.from_pretrained(
    "finetuned_model/chronos_extended/run-0/checkpoint-9000",
    device_map=device,
    dtype=torch.float32,
)

pred_ext = evaluate_chronos(
    extended,
    context,
    actual,
    "Chronos Extended FT"
)


# =========================
# Moirai
# =========================

print("Moirai 로드 중...")

moirai = MoiraiForecast.from_pretrained(
    "Salesforce/moirai-1.1-R-small"
).to(device)

pred_moirai = evaluate_moirai(
    moirai,
    context,
    actual,
    "Moirai-1.1-R-small"
)


# =========================
# 최종 결과
# =========================

results = []

for name, pred in [
    ("Chronos Zero-shot", pred_zero),
    ("Chronos Baseline FT", pred_base),
    ("Chronos Extended FT", pred_ext),
    ("Moirai", pred_moirai),
]:

    mae = np.mean(np.abs(actual - pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100

    results.append([name, mae, mape])

results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "MAPE"]
)

print("\n========================")
print(results_df.sort_values("MAE"))
print("========================")